# Synthetic Document + Comment Thread Generator

This notebook builds a reproducible pipeline for generating synthetic documents and associated comment threads using a large language model (LLM). Each dataset entry mirrors the procedure described in the project specification.

## Procedure Overview

The generator retrieves one of ten predefined topics, creates a four-paragraph document, highlights a contentious sentence with the peer's first remark, and then alternates between author and peer comments until the thread reaches five peer inputs and four author responses.

In [1]:
import json
from src import llms, utils, task_conflict_generator

def create_mock_responder(prompt: str, system_prompt=None, max_output_tokens: int = 800) -> str:
    """Custom responder for mock testing."""
    if "Write a short document" in prompt:
        return (
            "Paragraph 1: This mock document introduces a controversial workplace policy.\n\n"
            "Paragraph 2: The policy will reduce budgets by 15 percent, even if teams object.\n\n"
            "Paragraph 3: Leaders believe sharper cuts will motivate better performance."
        )
    if "Respond with a JSON object" in prompt:
        return json.dumps({
            "highlighted_sentence": "The policy will reduce budgets by 15 percent, even if teams object.",
            "comment": "This sentence feels too dismissive of the teams' concerns; can we soften it?",
        })
    if "The author would like to keep the doc as it is" in prompt:
        return "I understand the concern, but the firm directive comes straight from leadership and we need to reflect that reality."
    if "The peer still disagrees" in prompt:
        return "We still need to flag that dismissing objections may alienate the staff; please acknowledge the risk."
    return "This is a placeholder response from the mock client."


def preview_mock_entry():
    """Generate a mock dataset entry for preview."""
    mock_llm = llms.MockLLMClient(responder=create_mock_responder)
    return task_conflict_generator.generate_conflict_context(mock_llm, selected_topic=task_conflict_generator._TOPICS[0])

In [2]:
mock_entry = preview_mock_entry()
print(json.dumps(mock_entry.to_dict(), indent=2))

{
  "topic": "a news report on a local incident",
  "document": "This is a placeholder response from the mock client.",
  "highlighted_sentence": "The policy will reduce budgets by 15 percent, even if teams object.",
  "comment_thread": [
    {
      "speaker": "peer",
      "text": "This sentence feels too dismissive of the teams' concerns; can we soften it?"
    },
    {
      "speaker": "author",
      "text": "This is a placeholder response from the mock client."
    },
    {
      "speaker": "peer",
      "text": "This is a placeholder response from the mock client."
    },
    {
      "speaker": "author",
      "text": "This is a placeholder response from the mock client."
    },
    {
      "speaker": "peer",
      "text": "This is a placeholder response from the mock client."
    }
  ]
}


In [3]:
llm = llms.OpenAiClient(model='gpt-4o-mini', temperature=0.8)
generated_data = task_conflict_generator.generate_conflict_context_dataset(llm, 1)
print(json.dumps(generated_data[0].to_dict(), indent=2))

{
  "topic": "a news report on a local incident",
  "document": "**Local Incident Report: Disturbance at Riverside Park**\n\nOn the evening of October 18, a chaotic scene unfolded at Riverside Park as multiple eyewitnesses reported a large brawl involving dozens of individuals. This incident, which began around 7 PM, drew the attention of both local authorities and concerned residents. The altercation, reportedly fueled by an ongoing feud between rival groups, quickly escalated, prompting onlookers to call for police assistance. Officers arrived on the scene shortly thereafter, managing to disperse the crowd and restore order, but not before several participants sustained injuries that required medical attention.\n\nWhat\u2019s truly disheartening about this incident is not just the violence itself, but the broader implications it has for our community. Riverside Park, a space that should serve as a haven for families and individuals seeking respite, has now become a backdrop for disor

In [4]:
utils.save_dataset_to_jsonl(generated_data, './results/docs.jsonl')